In [25]:
from dotenv import load_dotenv

load_dotenv()

True

In [26]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favorite_colour: str

# Write to state

In [27]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

In [28]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [29]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is Red")]},
    {"configurable": {"thread_id": "1"}}
)

In [30]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is Red', additional_kwargs={}, response_metadata={}, id='794eb91c-2eaa-482b-8a19-a95461f5a4ca'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:50.564245Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1437061000, 'load_duration': 145582792, 'prompt_eval_count': 165, 'prompt_eval_duration': 277477000, 'eval_count': 24, 'eval_duration': 1007090000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c8-fa84-7b31-aa5f-e35ce9e1e890-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'Red'}, 'id': '83de4e61-2cb7-483b-b2df-c5fc4383f9e2', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 165, 'output_tokens': 24, 'total_tokens': 189}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='b7

In [31]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    }, # type: ignore
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='1b08e9f0-e0ad-49e1-adbb-63edfbd58446'),
              AIMessage(content="I'm here to assist you better. How can I help you today?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:52.448554Z', 'done': True, 'done_reason': 'stop', 'total_duration': 920500125, 'load_duration': 148506750, 'prompt_eval_count': 166, 'prompt_eval_duration': 150912000, 'eval_count': 16, 'eval_duration': 615140000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c9-03e2-7c22-a313-5b7bf628b263-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 166, 'output_tokens': 16, 'total_tokens': 182})]}


# Read State

In [32]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    model="ollama:qwen2.5:3b",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [33]:

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is Red")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is Red', additional_kwargs={}, response_metadata={}, id='d01353a0-3e18-47fe-9a0e-da083c613833'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:54.00281Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1476971542, 'load_duration': 142328917, 'prompt_eval_count': 202, 'prompt_eval_duration': 382465000, 'eval_count': 24, 'eval_duration': 949725000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c9-07cc-7840-852b-43cc47f298ff-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'Red'}, 'id': '1e7c0b85-a9b0-4b5e-8970-5354cdb415b5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 24, 'total_tokens': 226}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='1bc9

In [34]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content='My favourite colour is Red', additional_kwargs={}, response_metadata={}, id='d01353a0-3e18-47fe-9a0e-da083c613833'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T17:59:54.00281Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1476971542, 'load_duration': 142328917, 'prompt_eval_count': 202, 'prompt_eval_duration': 382465000, 'eval_count': 24, 'eval_duration': 949725000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f61c9-07cc-7840-852b-43cc47f298ff-0', tool_calls=[{'name': 'update_favourite_colour', 'args': {'favourite_colour': 'Red'}, 'id': '1e7c0b85-a9b0-4b5e-8970-5354cdb415b5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 202, 'output_tokens': 24, 'total_tokens': 226}),
              ToolMessage(content='Successfully updated favourite colour', name='update_favourite_colour', id='1bc9

In [36]:
print(response["messages"][-1].content)

It seems like there might have been an issue fetching your favourite color. Could you please provide it again?
